In [ ]:
!pip install openai llama-index yt-dlp gradio moviepy SpeechRecognition lancedb llama-index-vector-stores-lancedb llama-index-multi-modal-llms-openai

In [ ]:
!pip install -U setuptools
!pip install -U moviepy

In [ ]:
!pip show openai-whisper
!pip show whisper
!pip show llama-index-multi-modal-llms-openai

In [ ]:
!apt install -y ffmpeg
!pip install git+https://github.com/openai/whisper.git
!pip install llama-index-embeddings-clip
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
!pip install langdetect

In [ ]:
!pip install -U yt-dlp

In [ ]:
import os
import shutil
import json
import sys
import uuid
import time
import yt_dlp
import gradio as gr
from moviepy import VideoFileClip
from PIL import Image
from langdetect import detect

# OpenAI Client
from openai import OpenAI

# LlamaIndex Components
from llama_index.core import SimpleDirectoryReader, StorageContext, Document
from llama_index.core.indices import MultiModalVectorStoreIndex
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.core.schema import ImageNode
from llama_index.multi_modal_llms.openai import OpenAIMultiModal

# =========================================================
# CONFIGURATION
# =========================================================

# API key must be set as an environment variable. No hardcoded fallback.
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("\u274c Error: OPENAI_API_KEY environment variable is not set.")
    sys.exit(1)

os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

# Folder paths
VIDEO_DIR = "./video_data"
MIXED_DIR = "./mixed_data"
LANCEDB_DIR = "./lancedb"

# Processing settings
MAX_FRAMES = 30       # Number of snapshots to take from the video
IMG_SIZE = (512, 288) # Low res for faster processing

# =========================================================
# SYSTEM CLEANUP
# =========================================================

def reset_environment():
    """Cleans up temporary folders to prevent data mixing between runs."""
    for d in [VIDEO_DIR, MIXED_DIR]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

# =========================================================
# VIDEO DOWNLOAD
# =========================================================

def download_video(url):
    """Downloads a video from YouTube using yt-dlp."""
    print(f"Downloading: {url}")

    cookies_path = "/content/cookies.txt"

    ydl_opts = {
        "outtmpl": f"{VIDEO_DIR}/input_vid.%(ext)s",
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4/best[ext=mp4]/best",
        "quiet": True,
        "no_warnings": True,
        "nocheckcertificate": True,
        "user_agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        ),
    }

    if os.path.exists(cookies_path):
        ydl_opts["cookiefile"] = cookies_path
    else:
        print(
            " No cookies.txt found at /content/cookies.txt — "
            "YouTube may block this request. Export cookies.txt from a "
            "logged-in browser session and upload it before retrying."
        )

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            return {
                "title": info.get("title", "Unknown"),
                "uploader": info.get("uploader", "Unknown"),
                "duration": info.get("duration", 0),
                "view_count": info.get("view_count", 0),
            }
    except Exception as e:
        import traceback
        error_msg = str(e)
        print("\n DOWNLOAD ERROR\n")
        traceback.print_exc()

        if "Sign in to confirm" in error_msg or "cookies" in error_msg.lower():
            raise ValueError(
                "YouTube requires authentication for this video. "
                "Upload a fresh cookies.txt (exported while logged into "
                "youtube.com) to /content/cookies.txt and try again."
            )

        raise ValueError(f"Failed to download video: {error_msg}")
# =========================================================
# VISUAL PROCESSING (FRAMES)
# =========================================================

def extract_frames(video_path, max_frames=MAX_FRAMES):
    """Extracts evenly spaced frames from the video."""
    print("\U0001f4f8 Extracting frames...")
    try:
        clip = VideoFileClip(video_path)
        duration = clip.duration

        if duration <= 0:
            return

        timestamps = [i * (duration / max_frames) for i in range(max_frames)]

        for idx, t in enumerate(timestamps):
            frame = clip.get_frame(t)
            img = Image.fromarray(frame)
            img = img.resize(IMG_SIZE)
            img.save(os.path.join(MIXED_DIR, f"frame_{idx:03d}.jpg"), quality=80)

        clip.close()
    except Exception as e:
        print(f"\u26a0\ufe0f Error extracting frames: {e}")

# =========================================================
# AUDIO PROCESSING (WHISPER, AUTO LANGUAGE DETECTION)
# =========================================================

def process_audio(video_path):
    """Extracts audio and transcribes it with OpenAI Whisper (auto language detection)."""
    print("🎤 Extracting and transcribing audio...")
    audio_path = os.path.join(MIXED_DIR, "audio.mp3")

    try:
        clip = VideoFileClip(video_path)
        clip.audio.write_audiofile(
            audio_path,
            logger=None,
            bitrate="32k"
        )
        clip.close()

        with open(audio_path, "rb") as audio_file:
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file
            )

        text = transcript.text

        # Detect transcript language (debugging only)
        try:
            transcript_lang = detect(text[:5000])
        except Exception:
            transcript_lang = "unknown"

        print(f"Transcript Language: {transcript_lang}")
        print(f"Transcription complete. Length: {len(text)} chars")

        return text

    except Exception as e:
        print(f"Audio processing failed: {e}")
        return ""

# =========================================================
# INDEXING (MULTIMODAL RAG PIPELINE)
# =========================================================

def build_index(video_url):
    """Main pipeline: download -> extract -> index."""
    reset_environment()

    metadata = download_video(video_url)

    video_file = next(
        (os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR) if f.startswith("input_vid")),
        None
    )
    if not video_file:
        raise FileNotFoundError("Video download failed.")

    extract_frames(video_file)
    transcript_text = process_audio(video_file)

    if transcript_text:
        with open(os.path.join(MIXED_DIR, "transcript.txt"), "w", encoding="utf-8") as f:
            f.write(transcript_text)

    documents = SimpleDirectoryReader(MIXED_DIR).load_data()

    # Unique table names per run to avoid collisions across sessions
    table_suffix = str(uuid.uuid4())[:8]

    text_store = LanceDBVectorStore(uri=LANCEDB_DIR, table_name=f"text_{table_suffix}")
    image_store = LanceDBVectorStore(uri=LANCEDB_DIR, table_name=f"imgs_{table_suffix}")

    storage_context = StorageContext.from_defaults(
        vector_store=text_store,
        image_store=image_store
    )

    print("\u2699\ufe0f Building vector index...")
    index = MultiModalVectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context
    )

    return metadata, index, video_file

# =========================================================
# QUERY ENGINE (MULTILINGUAL)
# =========================================================

def ask_engine(query, metadata, index):
    """
    Multilingual Video RAG
    Answer language = User Question Language
    Video language can be anything.
    """

    if not index:
        return " Please process a video first.", []

    # =====================================
    # Detect User Question Language
    # =====================================

    try:
        detected_lang = detect(query)
    except:
        detected_lang = "en"

    LANG_MAP = {
        "en": "English",
        "hi": "Hindi",
        "te": "Telugu",
        "ta": "Tamil",
        "kn": "Kannada",
        "ml": "Malayalam",
        "mr": "Marathi",
        "gu": "Gujarati",
        "bn": "Bengali",
        "pa": "Punjabi",
        "ur": "Urdu",
        "es": "Spanish",
        "fr": "French",
        "de": "German",
        "it": "Italian",
        "pt": "Portuguese",
        "ru": "Russian",
        "ja": "Japanese",
        "ko": "Korean",
        "zh-cn": "Chinese",
        "zh-tw": "Chinese"
    }

    answer_language = LANG_MAP.get(detected_lang, "English")

    print(f" Question Language: {answer_language}")

    # =====================================
    # Retrieval
    # =====================================

    retriever = index.as_retriever(
        similarity_top_k=3,
        image_similarity_top_k=3
    )

    retrieval_results = retriever.retrieve(query)

    retrieved_images = []
    retrieved_texts = []

    for res in retrieval_results:

        if isinstance(res.node, ImageNode):

            path = res.node.metadata.get("file_path")

            if path and os.path.exists(path):
                retrieved_images.append(path)

        else:
            retrieved_texts.append(res.text)

    context_str = "\n\n".join(retrieved_texts)

    if not context_str.strip():
        context_str = "No transcript context retrieved."

    image_nodes = [
        ImageNode(image_path=p)
        for p in retrieved_images
    ]

    # =====================================
    # System Prompt
    # =====================================

    prompt = f"""
You are an expert multilingual video analysis assistant.

IMPORTANT RULES:

1. The user's language is {answer_language}.

2. You MUST answer ONLY in {answer_language}.

3. The video transcript and retrieved context may be in ANY language.

4. Use the transcript only as a source of information.

5. Translate internally if necessary.

6. NEVER copy the transcript language unless it matches {answer_language}.

7. Use both transcript context and visual evidence.

8. If the answer is not available in the context, clearly say so in {answer_language}.

VIDEO METADATA:
{json.dumps(metadata, ensure_ascii=False)}

TRANSCRIPT CONTEXT:
{context_str}

USER QUESTION:
{query}
"""

    # =====================================
    # GPT-4o Multimodal
    # =====================================

    llm = OpenAIMultiModal(
        model="gpt-4o",
        api_key=os.environ["OPENAI_API_KEY"],
        max_new_tokens=1024
    )

    response = llm.complete(
        prompt=prompt,
        image_documents=image_nodes
    )

    return response.text, retrieved_images

In [ ]:
def create_ui():
    with gr.Blocks(title="Multilingual Video AI") as app:
        gr.Markdown("# Universal Video QA System")
        gr.Markdown("Paste any YouTube video URL (any language) and ask questions in **any language**.")

        state_idx = gr.State(None)
        state_meta = gr.State(None)

        with gr.Row():
            url_input = gr.Textbox(label="YouTube URL", placeholder="https://youtube.com/...", scale=4)
            process_btn = gr.Button("Process Video", scale=1, variant="primary")

        status_output = gr.Markdown("Waiting for input...")

        with gr.Row():
            vid_display = gr.Video(label="Input Video", height=300)
            meta_display = gr.JSON(label="Metadata")

        gr.Markdown("### Ask a Question")
        query_input = gr.Textbox(label="Question", placeholder="e.g. What is the main topic of this video?", lines=2)
        ask_btn = gr.Button("Ask AI", variant="primary")

        with gr.Row():
            answer_output = gr.Textbox(label="AI Answer", interactive=False, lines=8)
            gallery_output = gr.Gallery(label="Visual Evidence", columns=2, height=300)

        def on_process(url):
            try:
                meta, idx, vid_path = build_index(url)
                return (
                    idx, meta,
                    vid_path, meta,
                    "\u2705 Video processed successfully! Ready for questions."
                )
            except Exception as e:
                return (
                    None, None,
                    None, None,
                    f"\u274c Error: {str(e)}"
                )

        process_btn.click(
            on_process,
            inputs=[url_input],
            outputs=[state_idx, state_meta, vid_display, meta_display, status_output]
        )

        def on_ask(q, idx, meta):
            return ask_engine(q, meta, idx)

        ask_btn.click(
            on_ask,
            inputs=[query_input, state_idx, state_meta],
            outputs=[answer_output, gallery_output]
        )

    return app

if __name__ == "__main__":
    reset_environment()
    demo = create_ui()
    demo.launch(share=True)
